In [1]:
from pyspark.sql.functions import (
    col,
    to_date,
    sum as spark_sum,
    count,
    countDistinct
)

# Read the validated Silver table
gold_source = spark.table("silver.sales_transactions")

# Keep only completed sales for the daily sales Gold table
sales_only = gold_source.filter(
    col("TransactionType") == "Sale"
)

# Build daily sales summary
gold_daily_sales = (
    sales_only
    .withColumn("SalesDate", to_date(col("InvoiceDate")))
    .groupBy("SalesDate")
    .agg(
        spark_sum("LineAmount").alias("TotalRevenue"),
        spark_sum("Quantity").alias("TotalQuantity"),
        countDistinct("InvoiceNo").alias("OrderCount"),
        count("*").alias("LineItemCount"),
        countDistinct("CustomerID").alias("CustomerCount")
    )
    .orderBy("SalesDate")
)

gold_daily_sales.show(20, truncate=False)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 3, Finished, Available, Finished, False)

+----------+------------+-------------+----------+-------------+-------------+
|SalesDate |TotalRevenue|TotalQuantity|OrderCount|LineItemCount|CustomerCount|
+----------+------------+-------------+----------+-------------+-------------+
|2010-12-01|58776.7900  |26906        |136       |3037         |96           |
|2010-12-02|47629.4200  |31283        |143       |2023         |100          |
|2010-12-03|46898.6300  |16430        |73        |2143         |51           |
|2010-12-05|31364.6300  |16243        |88        |2604         |76           |
|2010-12-06|54624.1500  |21775        |108       |3774         |83           |
|2010-12-07|99553.8500  |25324        |85        |2904         |66           |
|2010-12-08|45235.3600  |23049        |123       |2533         |100          |
|2010-12-09|53548.1900  |20698        |132       |2791         |91           |
|2010-12-10|59021.0200  |21031        |81        |2663         |54           |
|2010-12-12|17125.6500  |10537        |44        |13

In [2]:
print("Gold daily rows:", gold_daily_sales.count())

print(
    "Duplicate SalesDate rows:",
    gold_daily_sales.groupBy("SalesDate")
    .count()
    .filter(col("count") > 1)
    .count()
)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 4, Finished, Available, Finished, False)

Gold daily rows: 305
Duplicate SalesDate rows: 0


In [3]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

(
    gold_daily_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.daily_sales")
)

print("gold.daily_sales created successfully.")

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 5, Finished, Available, Finished, False)

gold.daily_sales created successfully.


In [4]:
from pyspark.sql.functions import (
    col,
    when,
    sum as spark_sum,
    countDistinct
)

product_source = gold_source.filter(
    col("TransactionType").isin("Sale", "Cancellation")
)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 6, Finished, Available, Finished, False)

In [5]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

description_window = (
    Window
    .partitionBy("StockCode")
    .orderBy(col("InvoiceDate").desc())
)

latest_description = (
    product_source
    .filter(col("Description").isNotNull())
    .withColumn(
        "rn",
        row_number().over(description_window)
    )
    .filter(col("rn") == 1)
    .select(
        "StockCode",
        col("Description").alias("CanonicalDescription")
    )
)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 7, Finished, Available, Finished, False)

In [6]:
product_metrics = (
    product_source
    .groupBy("StockCode")
    .agg(
        spark_sum(
            when(col("TransactionType") == "Sale", col("Quantity"))
            .otherwise(0)
        ).alias("GrossQuantitySold"),

        spark_sum(
            when(col("TransactionType") == "Cancellation", col("Quantity"))
            .otherwise(0)
        ).alias("CancelledQuantity"),

        spark_sum(
            when(col("TransactionType") == "Sale", col("LineAmount"))
            .otherwise(0)
        ).alias("GrossRevenue"),

        spark_sum(
            when(col("TransactionType") == "Cancellation", col("LineAmount"))
            .otherwise(0)
        ).alias("CancellationValue"),

        countDistinct(
            when(col("TransactionType") == "Sale", col("InvoiceNo"))
        ).alias("OrderCount"),

        countDistinct(
            when(col("TransactionType") == "Sale", col("CustomerID"))
        ).alias("CustomerCount")
    )
    .withColumn(
        "NetQuantity",
        col("GrossQuantitySold") + col("CancelledQuantity")
    )
    .withColumn(
        "NetRevenue",
        col("GrossRevenue") + col("CancellationValue")
    )
)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 8, Finished, Available, Finished, False)

In [7]:
gold_product_performance = (
    product_metrics
    .join(
        latest_description,
        on="StockCode",
        how="left"
    )
)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 9, Finished, Available, Finished, False)

In [8]:
print("Product rows:", gold_product_performance.count())

print(
    "Duplicate StockCodes:",
    gold_product_performance.groupBy("StockCode")
    .count()
    .filter(col("count") > 1)
    .count()
)

gold_product_performance.select(
    "StockCode",
    "CanonicalDescription",
    "GrossQuantitySold",
    "CancelledQuantity",
    "NetQuantity",
    "GrossRevenue",
    "CancellationValue",
    "NetRevenue",
    "OrderCount",
    "CustomerCount"
).orderBy(
    col("NetRevenue").desc()
).show(20, truncate=False)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 10, Finished, Available, Finished, False)

Product rows: 3956
Duplicate StockCodes: 0
+---------+----------------------------------+-----------------+-----------------+-----------+------------+-----------------+-----------+----------+-------------+
|StockCode|CanonicalDescription              |GrossQuantitySold|CancelledQuantity|NetQuantity|GrossRevenue|CancellationValue|NetRevenue |OrderCount|CustomerCount|
+---------+----------------------------------+-----------------+-----------------+-----------+------------+-----------------+-----------+----------+-------------+
|DOT      |DOTCOM POSTAGE                    |1708             |-1               |1707       |206248.7700 |-3.2900          |206245.4800|709       |2            |
|22423    |REGENCY CAKESTAND 3 TIER          |13860            |-855             |13005      |174131.0400 |-9697.0500       |164433.9900|1987      |880          |
|47566    |PARTY BUNTING                     |18287            |-277             |18010      |99445.2300  |-1201.3500       |98243.8800 |1686 

In [9]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

(
    gold_product_performance.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.product_performance")
)

print("gold.product_performance created successfully.")

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 11, Finished, Available, Finished, False)

gold.product_performance created successfully.


In [10]:
from pyspark.sql.functions import (
    col,
    when,
    sum as spark_sum,
    countDistinct,
    min as spark_min,
    max as spark_max
)

# Customer-level analysis requires a known CustomerID
customer_source = gold_source.filter(
    col("CustomerID").isNotNull()
)


StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 12, Finished, Available, Finished, False)

In [11]:
gold_customer_summary = (
    customer_source
    .groupBy("CustomerID")
    .agg(
        spark_min(
            when(
                col("TransactionType") == "Sale",
                col("InvoiceDate")
            )
        ).alias("FirstPurchaseDate"),

        spark_max(
            when(
                col("TransactionType") == "Sale",
                col("InvoiceDate")
            )
        ).alias("LastPurchaseDate"),

        countDistinct(
            when(
                col("TransactionType") == "Sale",
                col("InvoiceNo")
            )
        ).alias("OrderCount"),

        spark_sum(
            when(
                col("TransactionType") == "Sale",
                col("Quantity")
            ).otherwise(0)
        ).alias("TotalQuantity"),

        spark_sum(
            when(
                col("TransactionType") == "Sale",
                col("LineAmount")
            ).otherwise(0)
        ).alias("GrossRevenue"),

        spark_sum(
            when(
                col("TransactionType") == "Cancellation",
                col("LineAmount")
            ).otherwise(0)
        ).alias("CancellationValue"),

        countDistinct(
            when(
                col("TransactionType") == "Sale",
                col("StockCode")
            )
        ).alias("DistinctProductsBought")
    )
    .withColumn(
        "NetRevenue",
        col("GrossRevenue") + col("CancellationValue")
    )
    .withColumn(
        "AverageOrderValue",
        when(
            col("OrderCount") > 0,
            col("GrossRevenue") / col("OrderCount")
        )
    )
    .withColumn(
        "NetAverageOrderValue",
        when(
            col("OrderCount") > 0,
            col("NetRevenue") / col("OrderCount")
        )
    )
)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 13, Finished, Available, Finished, False)

In [12]:
print("Customer rows:", gold_customer_summary.count())

print(
    "Duplicate CustomerIDs:",
    gold_customer_summary.groupBy("CustomerID")
    .count()
    .filter(col("count") > 1)
    .count()
)

gold_customer_summary.orderBy(
    col("NetRevenue").desc()
).show(20, truncate=False)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 14, Finished, Available, Finished, False)

Customer rows: 4373
Duplicate CustomerIDs: 0
+----------+-------------------+-------------------+----------+-------------+------------+-----------------+----------------------+------------+-----------------+--------------------+
|CustomerID|FirstPurchaseDate  |LastPurchaseDate   |OrderCount|TotalQuantity|GrossRevenue|CancellationValue|DistinctProductsBought|NetRevenue  |AverageOrderValue|NetAverageOrderValue|
+----------+-------------------+-------------------+----------+-------------+------------+-----------------+----------------------+------------+-----------------+--------------------+
|0         |2010-12-01 11:52:00|2011-12-09 10:26:00|2184      |477620       |1741759.6800|-285290.2600     |3543                  |1456469.4200|797.509011       |666.881603          |
|14646     |2010-12-20 10:09:00|2011-12-08 12:12:00|74        |197491       |280206.0200 |-717.0000        |701                   |279489.0200 |3786.567838      |3776.878649         |
|18102     |2010-12-07 16:42:00|201

In [13]:
(
    gold_customer_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.customer_summary")
)

print("gold.customer_summary created successfully.")

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 15, Finished, Available, Finished, False)

gold.customer_summary created successfully.


In [16]:
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    countDistinct,
    when
)

# Completed sales only
country_sales_source = gold_source.filter(
    col("TransactionType") == "Sale"
)

gold_country_sales = (
    country_sales_source
    .groupBy("Country")
    .agg(
        spark_sum("LineAmount").alias("TotalRevenue"),
        spark_sum("Quantity").alias("TotalQuantity"),
        countDistinct("InvoiceNo").alias("OrderCount"),
        countDistinct("CustomerID").alias("CustomerCount")
    )
    .withColumn(
        "AverageOrderValue",
        when(
            col("OrderCount") > 0,
            col("TotalRevenue") / col("OrderCount")
        )
    )
)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 18, Finished, Available, Finished, False)

In [17]:
print("Country rows:", gold_country_sales.count())

print(
    "Duplicate Countries:",
    gold_country_sales.groupBy("Country")
    .count()
    .filter(col("count") > 1)
    .count()
)

gold_country_sales.orderBy(
    col("TotalRevenue").desc()
).show(50, truncate=False)

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 19, Finished, Available, Finished, False)

Country rows: 37
Duplicate Countries: 0
+--------------------+------------+-------------+----------+-------------+-----------------+
|Country             |TotalRevenue|TotalQuantity|OrderCount|CustomerCount|AverageOrderValue|
+--------------------+------------+-------------+----------+-------------+-----------------+
|United Kingdom      |8990682.0340|4718326      |18783     |3922         |478.660599       |
|Netherlands         |285446.3400 |200937       |95        |9            |3004.698316      |
|Ireland             |283140.5200 |147281       |288       |4            |983.126806       |
|Germany             |228678.4000 |119156       |457       |94           |500.390372       |
|France              |209625.3700 |112061       |392       |88           |534.758597       |
|Australia           |138453.8100 |84199        |57        |9            |2429.014211      |
|Spain               |61558.5600  |27944        |90        |30           |683.984000       |
|Switzerland         |57067.60

In [18]:
(
    gold_country_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold.country_sales")
)

print("gold.country_sales created successfully.")

StatementMeta(, 8e295346-8d0e-4867-a3c4-5c6a982387ad, 20, Finished, Available, Finished, False)

gold.country_sales created successfully.
